# GHSL — population quickstart

Download GHS-POP 2020 at 100 m over a small Lisbon AOI and map the residential
population grid. The Mollweide source tile is reprojected to WGS84 and cropped
to the bbox by `pyramids`.

## Setup

Consolidate the imports. `earthlens` provides the unified `EarthLens` entry
point; `pyramids` provides `Dataset` for reading the written GeoTIFF; and we
use a temporary directory as the download target.

In [ ]:
import tempfile

import numpy as np
from pyramids.dataset import Dataset

from earthlens.core import EarthLens

## Download GHS-POP

Build the request first — source, variable, year window, and a small bounding
box over Lisbon — pointing the output at a temporary directory.

In [ ]:
out = tempfile.mkdtemp()
ghsl = EarthLens(
    data_source='ghsl',
    variables=['GHS_POP'],
    start='2020-01-01',
    end='2020-12-31',
    aoi=[-9.20, 38.68, -9.10, 38.78],
    path=out,
)

Run the download. GHSL is a raster backend, so `download()` returns the list of
written GeoTIFF paths.

In [ ]:
paths = ghsl.download(progress_bar=False)
paths

## Read and plot the grid

Read the written GeoTIFF with `pyramids`, collapse to a 2D array, and mask the
no-data cells (negative values) to `NaN`.

In [ ]:
ds = Dataset.read_file(str(paths[0]))
arr = np.asarray(ds.read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr < 0, np.nan, arr)
print('epsg', ds.epsg, '| shape', arr.shape)

Map the residential population grid (people per 100 m cell).

In [ ]:
# plot() places the raster on georeferenced axes and honours the band's
# declared no-data, so nothing is read or masked for the picture.
glyph = ds.plot(cmap='magma', title='GHS-POP 2020 (100 m) — Lisbon')
glyph.cbar.set_label('people / cell')

## Headline number

A quick total — the number of residents inside the AOI.

In [ ]:
print(f'total population in AOI: {np.nansum(arr):,.0f}')